# 1. Tổng quan dữ liệu

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Tự tìm dataset từ thư mục notebook
current = Path.cwd().resolve()

DATA_PATH = None
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

for folder in SEARCH_ROOTS:
    candidates = [
        folder / "data" / "raw" / "RT_IOT2022.csv",
        folder / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
        folder / "final_IoT" / "sample_project" / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError("Khong tim thay RT_IOT2022.csv trong cau truc project")

df = pd.read_csv(DATA_PATH)

# Lưu các file kết quả cùng thư mục với notebook
OUTPUT_DIR = DATA_PATH.parents[2] / "result" / "eda" / "result_01_data_overview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Kích thước dữ liệu:", df.shape)
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("Thư mục lưu kết quả:", OUTPUT_DIR)

display(df.head())

# 2. Kiểm tra cấu trúc dữ liệu

In [ ]:
print("=== THÔNG TIN DATASET ===")
df.info()

print("\n=== THỐNG KÊ KIỂU DỮ LIỆU ===")
print(df.dtypes.value_counts())

print("\n=== DANH SÁCH TẤT CẢ THUỘC TÍNH ===")

columns_info = pd.DataFrame({
    "STT": range(1, len(df.columns) + 1),
    "Tên thuộc tính": df.columns,
    "Kiểu dữ liệu": df.dtypes.astype(str).values,
    "Số giá trị khác nhau": df.nunique().values
})

display(columns_info)

# 3. Kiểm tra chất lượng của dữ liệu

In [ ]:
print("=== GIÁ TRỊ THIẾU ===")
print("Tổng giá trị thiếu:", df.isnull().sum().sum())

print("\n=== GIÁ TRỊ VÔ HẠN ===")

numeric_df = df.select_dtypes(include=np.number)

print("Tổng giá trị Inf:",
      np.isinf(numeric_df.to_numpy()).sum())

print("\n=== DỮ LIỆU TRÙNG LẶP ===")

# Kiểm tra toàn bộ các cột
print("Số dòng trùng toàn bộ:",
      df.duplicated().sum())

# Không sử dụng cột chỉ số trong phép so sánh
feature_cols = df.columns.drop("Unnamed: 0", errors="ignore")

print("Số dòng trùng khi bỏ Unnamed: 0:",
      df.duplicated(subset=feature_cols).sum())

print("\n=== CÁC CỘT CHỈ CÓ MỘT GIÁ TRỊ ===")

constant_cols = df.columns[df.nunique(dropna=False) == 1]

print(constant_cols.tolist())

# 4. Thống kê mô tả các đặc trưng

In [ ]:
# Không đưa cột chỉ số vào phân tích
numeric_df = df.select_dtypes(include=np.number).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Tính thống kê mô tả
stats = numeric_df.describe().T

stats = stats.rename(columns={
    "count": "Count",
    "mean": "Mean",
    "std": "Std",
    "min": "Min",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Max"
})

# Bổ sung chỉ số để phát hiện vấn đề
stats["Skewness"] = numeric_df.skew()
stats["Zero_Percent"] = (
    numeric_df.eq(0).mean() * 100
)
stats["Unique_Values"] = numeric_df.nunique()

print("=== THỐNG KÊ MÔ TẢ ===")
display(stats.round(3))

# 1. Các thuộc tính có phân phối lệch mạnh
skewed = stats[
    stats["Skewness"].abs() > 3
].sort_values("Skewness", key=lambda x: x.abs(), ascending=False)

print("\n=== THUỘC TÍNH CÓ |SKEWNESS| > 3 ===")
print("Số lượng:", len(skewed))
display(
    skewed[
        ["Mean", "Median", "Std", "Skewness"]
    ].head(15).round(3)
)

# 2. Các thuộc tính chứa nhiều giá trị 0
zero_heavy = stats[
    stats["Zero_Percent"] >= 95
].sort_values("Zero_Percent", ascending=False)

print("\n=== THUỘC TÍNH CÓ >= 95% GIÁ TRỊ BẰNG 0 ===")
print("Số lượng:", len(zero_heavy))
display(
    zero_heavy[
        ["Min", "Median", "Max", "Zero_Percent"]
    ].round(3)
)

# 3. Các thuộc tính chỉ có một giá trị
constant = stats[
    stats["Unique_Values"] == 1
]

print("\n=== THUỘC TÍNH HẰNG ===")
display(constant[["Min", "Max", "Unique_Values"]])

# 5. Phân tích phân bố nhãn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target = "Attack_type"

class_counts = df[target].value_counts()

class_distribution = pd.DataFrame({
    "Số mẫu": class_counts,
    "Tỷ lệ (%)": (
        class_counts / len(df) * 100
    ).round(3)
})

print("=== PHÂN BỐ NHÃN ===")
display(class_distribution)

class_distribution.to_excel(
    OUTPUT_DIR / "class_distribution.xlsx",
    sheet_name="Phan_bo_nhan",
    index_label=target
)

print("\nSố lớp:", df[target].nunique())
print("Lớp nhiều nhất:", class_counts.idxmax())
print("Lớp ít nhất:", class_counts.idxmin())

imbalance_ratio = (
    class_counts.max() / class_counts.min()
)

print(f"Tỷ lệ mất cân bằng: {imbalance_ratio:.2f}:1")

# Vẽ biểu đồ
plt.figure(figsize=(12, 6))

sns.barplot(
    x=class_counts.values,
    y=class_counts.index,
    color="steelblue"
)

plt.title("Phân bố số lượng mẫu theo Attack_type")
plt.xlabel("Số lượng mẫu")
plt.ylabel("Loại tấn công")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "attack_type_distribution.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 6. Các vấn đề được phát hiện từ dataset

In [ ]:
import pandas as pd
import numpy as np

target = "Attack_type"

# Chỉ lấy đặc trưng số, không tính cột chỉ số
numeric_features = df.select_dtypes(
    include=np.number
).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# 1. Chất lượng dữ liệu
missing_count = int(df.isna().sum().sum())

inf_count = int(
    np.isinf(numeric_features.to_numpy()).sum()
)

duplicate_count = int(
    df.duplicated(
        subset=df.columns.drop(
            "Unnamed: 0",
            errors="ignore"
        )
    ).sum()
)

# 2. Thuộc tính hằng
constant_cols = df.columns[
    df.nunique(dropna=False) == 1
].tolist()

# 3. Phân phối lệch
skewness = numeric_features.skew()

skewed_cols = skewness[
    skewness.abs() > 3
].index.tolist()

# 4. Nhiều giá trị bằng 0
zero_percentage = (
    numeric_features.eq(0).mean() * 100
)

high_zero_cols = zero_percentage[
    zero_percentage >= 95
].index.tolist()

# 5. Mất cân bằng nhãn
class_counts = df[target].value_counts()

imbalance_ratio = (
    class_counts.max() / class_counts.min()
)

# Tổng hợp vào bảng
issues = pd.DataFrame({
    "Vấn đề": [
        "Giá trị thiếu",
        "Giá trị vô hạn",
        "Dữ liệu trùng lặp",
        "Thuộc tính hằng",
        "Phân phối lệch mạnh |Skew| > 3",
        "Thuộc tính có >= 95% giá trị 0",
        "Mất cân bằng nhãn"
    ],
    "Kết quả": [
        missing_count,
        inf_count,
        duplicate_count,
        len(constant_cols),
        len(skewed_cols),
        len(high_zero_cols),
        round(imbalance_ratio, 2)
    ],
    "Hướng nghiên cứu": [
        "Xem xét xử lý nếu có",
        "Xem xét xử lý nếu có",
        "Kiểm tra trùng lặp trước khi chia dữ liệu",
        "Đánh giá loại bỏ đặc trưng hằng",
        "Phân tích Histogram, KDE, Boxplot",
        "Đánh giá độ hữu ích của đặc trưng",
        "Phân tích lớp hiếm và phương pháp cân bằng"
    ]
})

print("=== TỔNG HỢP VẤN ĐỀ DATASET ===")
display(issues)

print("\n=== DANH SÁCH CỘT HẰNG ===")
print(constant_cols)

print("\n=== CÁC THUỘC TÍNH LỆCH MẠNH NHẤT ===")
display(
    skewness.abs()
    .sort_values(ascending=False)
    .head(10)
    .to_frame("Absolute Skewness")
)

print("\n=== CÁC THUỘC TÍNH NHIỀU GIÁ TRỊ 0 NHẤT ===")
display(
    zero_percentage
    .sort_values(ascending=False)
    .head(10)
    .to_frame("Zero (%)")
)

numeric_all = df.select_dtypes(include=np.number)
infinite_by_column = pd.Series(0, index=df.columns, dtype="int64")
infinite_by_column.loc[numeric_all.columns] = np.isinf(
    numeric_all.to_numpy()
).sum(axis=0)

overview = pd.DataFrame([
    {"Chỉ số": "Số dòng", "Giá trị": len(df)},
    {"Chỉ số": "Số cột", "Giá trị": len(df.columns)},
    {"Chỉ số": "Số cột số", "Giá trị": len(numeric_all.columns)},
    {"Chỉ số": "Số cột phân loại", "Giá trị": df.select_dtypes(exclude=np.number).shape[1]},
    {"Chỉ số": "Giá trị thiếu", "Giá trị": missing_count},
    {"Chỉ số": "Giá trị vô hạn", "Giá trị": inf_count},
    {"Chỉ số": "Dòng trùng (bỏ Unnamed: 0)", "Giá trị": duplicate_count},
    {"Chỉ số": "Số thuộc tính hằng", "Giá trị": len(constant_cols)},
    {"Chỉ số": "Thuộc tính hằng", "Giá trị": ", ".join(constant_cols)},
    {"Chỉ số": "Số lớp nhãn", "Giá trị": len(class_counts)},
    {"Chỉ số": "Lớp nhiều mẫu nhất", "Giá trị": class_counts.idxmax()},
    {"Chỉ số": "Số mẫu lớp nhiều nhất", "Giá trị": int(class_counts.max())},
    {"Chỉ số": "Lớp ít mẫu nhất", "Giá trị": class_counts.idxmin()},
    {"Chỉ số": "Số mẫu lớp ít nhất", "Giá trị": int(class_counts.min())},
    {"Chỉ số": "Tỷ lệ mất cân bằng", "Giá trị": f"{imbalance_ratio:.2f}:1"}
])

quality_summary = pd.DataFrame([
    {"Kiểm tra": "Giá trị thiếu", "Số lượng": missing_count, "Phạm vi": "Toàn bộ dataset"},
    {"Kiểm tra": "Giá trị vô hạn", "Số lượng": inf_count, "Phạm vi": "Các cột số"},
    {"Kiểm tra": "Dòng trùng", "Số lượng": duplicate_count, "Phạm vi": "Bỏ cột Unnamed: 0"},
    {"Kiểm tra": "Thuộc tính hằng", "Số lượng": len(constant_cols), "Phạm vi": ", ".join(constant_cols)}
])

quality_by_column = pd.DataFrame({
    "Thuộc tính": df.columns,
    "Giá trị thiếu": df.isna().sum().values,
    "Giá trị vô hạn": infinite_by_column.values
})

evidence_sheets = {
    "Tong_quan": overview,
    "Cau_truc_cot": columns_info,
    "Chat_luong": quality_summary,
    "Chat_luong_theo_cot": quality_by_column,
    "Thong_ke_so": stats.rename_axis("Thuộc tính").reset_index(),
    "Lech_manh": stats[stats["Skewness"].abs() > 3].rename_axis("Thuộc tính").reset_index(),
    "Nhieu_gia_tri_0": stats[stats["Zero_Percent"] >= 95].rename_axis("Thuộc tính").reset_index(),
    "Cot_hang": stats[stats["Unique_Values"] == 1].rename_axis("Thuộc tính").reset_index(),
    "Phan_bo_nhan": class_distribution.rename_axis(target).reset_index(),
    "Tong_hop_van_de": issues,
    "Top10_skew": skewness.abs().sort_values(ascending=False).head(10).rename("Absolute Skewness").rename_axis("Thuộc tính").reset_index(),
    "Top10_gia_tri_0": zero_percentage.sort_values(ascending=False).head(10).rename("Zero (%)").rename_axis("Thuộc tính").reset_index(),
    "Mau_5_dong": df.head().rename_axis("STT").reset_index()
}

evidence_path = OUTPUT_DIR / "01_Data_Overview_Evidence.xlsx"
with pd.ExcelWriter(evidence_path, engine="openpyxl") as writer:
    for sheet_name, table_data in evidence_sheets.items():
        table_data.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for column_cells in worksheet.columns:
            width = max(len(str(cell.value or "")) for cell in column_cells) + 2
            worksheet.column_dimensions[column_cells[0].column_letter].width = min(max(width, 10), 48)

print("\nĐã xuất workbook bằng chứng:", evidence_path)
